# 🍏 OptiVision: Control de Calidad Agroindustrial B2B
**Firma Consultora de IA**

Este Jupyter Notebook contiene el pipeline completo de Visión Artificial Clásica exigido para la demostración del sistema. Se omite el uso de Deep Learning (CNNs) en favor de métodos clásicos de extracción matemática y aprendizaje automático.

**Pipeline implementado:**
1. Extracción de características (HOG, LBP, GLCM) y Fusión.
2. Normalización mediante Z-score.
3. Clasificación y Benchmarking (k-NN, SVM, Random Forest).


In [1]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Descriptores matemáticos para la extracción de características
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay


## 1. Carga y Preprocesamiento del Dataset
**Nota de adaptación:** Las rutas locales absolutas han sido reemplazadas por rutas relativas (`./dataset_muestra/...`) para que el repositorio en GitHub sea reproducible por cualquier evaluador sin errores de ruta.


In [2]:
# --- RUTAS RELATIVAS PARA GITHUB ---
# Asegúrate de subir una carpeta llamada 'dataset_muestra' con las subcarpetas 'Frescas' y 'Podridas'
path_frescas = r"./dataset_muestra/Frescas"
path_podridas = r"./dataset_muestra/Podridas"
extensiones = ('.jpg', '.jpeg', '.png')

def cargar_imagenes_validas(path, limite=None):
    """
    Función optimizada para cargar las imágenes.
    Se redimensionan a 128x128 y se normalizan sobre la marcha
    para evitar saturar la memoria RAM.
    """
    # Verificar si la ruta existe para evitar errores en GitHub
    if not os.path.exists(path):
        print(f"ADVERTENCIA: La ruta {path} no existe. Crea las carpetas de muestra para ejecutar.")
        return np.array([])
        
    files = sorted([f for f in os.listdir(path) if f.lower().endswith(extensiones)])
    if limite is not None:
        files = files[:limite]
        
    imgs = []
    for f in files:
        img = cv2.imread(os.path.join(path, f))
        if img is not None:
            # Redimensionamos a 128x128 para estandarizar los datos de entrada
            img_resized = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (128, 128))
            imgs.append(img_resized.astype("float32") / 255.0)
    return np.array(imgs)

MAX_IMAGENES = 700 

print("Cargando imágenes de manzanas...")
img_frescas = cargar_imagenes_validas(path_frescas, limite=MAX_IMAGENES)
img_podridas = cargar_imagenes_validas(path_podridas, limite=MAX_IMAGENES)

if len(img_frescas) > 0 and len(img_podridas) > 0:
    print(f"Frescas cargadas: {len(img_frescas)} | Podridas cargadas: {len(img_podridas)}")
    
    # 0 para Podridas (defecto) y 1 para Frescas (óptimo)
    X = np.concatenate((img_podridas, img_frescas), axis=0)
    y = np.concatenate((np.zeros(len(img_podridas)), np.ones(len(img_frescas))), axis=0)
    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=40, stratify=y)
    print(f"Dataset listo -> Entrenamiento: {X_train.shape[0]} imágenes, Validación: {X_val.shape[0]} imágenes")
else:
    print("Dataset no encontrado. Se requieren imágenes para continuar la ejecución.")


Cargando imágenes de manzanas...
ADVERTENCIA: La ruta ./dataset_muestra/Frescas no existe. Crea las carpetas de muestra para ejecutar.
ADVERTENCIA: La ruta ./dataset_muestra/Podridas no existe. Crea las carpetas de muestra para ejecutar.
Dataset no encontrado. Se requieren imágenes para continuar la ejecución.


## 2. Extracción de Características (Feature Engineering)
Se extraen descriptores de forma (HOG), micro-textura (LBP) y contraste/homogeneidad (GLCM) para crear un vector de fusión robusto.


In [3]:
def extraer_hog(imagenes):
    hog_features = []
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        features = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
        hog_features.append(features)
    return np.array(hog_features)

def extraer_lbp(imagenes, P=8, R=1):
    lbp_features = []
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        lbp = local_binary_pattern(img_gray, P=P, R=R, method="uniform")
        (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
        hist = hist.astype("float32")
        hist /= (hist.sum() + 1e-7)
        lbp_features.append(hist)
    return np.array(lbp_features)

def extraer_glcm(imagenes, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    glcm_features = []
    propiedades = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    for img in imagenes:
        img_gray = cv2.cvtColor((img * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
        img_gray_q = (img_gray // 4).astype(np.uint8)
        glcm = graycomatrix(img_gray_q, distances=distances, angles=angles, levels=64, symmetric=True, normed=True)
        
        vector_img = []
        for prop in propiedades:
            valores = graycoprops(glcm, prop).ravel()
            vector_img.extend(valores)
        glcm_features.append(vector_img)
    return np.array(glcm_features)

if 'X_train' in locals():
    print("Extrayendo mapas de bordes (HOG)...")
    X_train_hog, X_val_hog = extraer_hog(X_train), extraer_hog(X_val)
    
    print("Extrayendo micro-texturas (LBP)...")
    X_train_lbp, X_val_lbp = extraer_lbp(X_train), extraer_lbp(X_val)
    
    print("Extrayendo coocurrencia espacial (GLCM)...")
    X_train_glcm, X_val_glcm = extraer_glcm(X_train), extraer_glcm(X_val)
    
    print("Generando Vector de Fusión...")
    X_train_fusion = np.hstack((X_train_hog, X_train_lbp, X_train_glcm))
    X_val_fusion = np.hstack((X_val_hog, X_val_lbp, X_val_glcm))


## 3. Entrenamiento, Normalización y Benchmarking Comercial
Se aplica Z-Score obligatorio y se comparan los modelos k-NN, SVM y Random Forest.


In [4]:
if 'X_train_fusion' in locals():
    # Normalización Z-Score
    scaler = StandardScaler()
    X_train_fusion_scaled = scaler.fit_transform(X_train_fusion)
    X_val_fusion_scaled = scaler.transform(X_val_fusion)
    
    models = {
        'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
        'SVM (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    }
    
    results = []
    print("Iniciando Benchmarking Comercial...\n")
    
    for model_name, model in models.items():
        t0 = time.time()
        model.fit(X_train_fusion_scaled, y_train)
        y_pred = model.predict(X_val_fusion_scaled)
        t_exec = time.time() - t0
        
        results.append({
            'Modelo': model_name,
            'Accuracy': accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred, average='binary'),
            'Recall': recall_score(y_val, y_pred, average='binary'),
            'F1-Score': f1_score(y_val, y_pred, average='binary'),
            'Tiempo (s)': t_exec
        })
    
    df_results = pd.DataFrame(results)
    print("TABLA DE MÉTRICAS B2B (Fusión HOG + LBP + GLCM)")
    print(df_results.to_string(index=False))


## 4. Visualización de Resultados Financieros e Industriales


In [5]:
if 'df_results' in locals():
    # 1. Gráfica de Eficiencia
    plt.figure(figsize=(9, 5))
    colores = ['#4C72B0', '#55A868', '#C44E52']
    plt.bar(df_results['Modelo'], df_results['F1-Score'], color=colores, edgecolor='black')
    plt.title("Benchmarking: Eficiencia en Clasificación de Manzanas", fontsize=13)
    plt.ylabel("F1-Score (Precisión equilibrada)", fontsize=11)
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    for i, v in enumerate(df_results['F1-Score']):
        plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # 2. Gráfica de Tiempos
    plt.figure(figsize=(9, 5))
    plt.bar(df_results['Modelo'], df_results['Tiempo (s)'], color=colores, edgecolor='black')
    plt.title("Latencia del Sistema (Tiempo de procesamiento del lote)", fontsize=13, fontweight='bold')
    plt.ylabel("Segundos (Menos es mejor)", fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    for i, v in enumerate(df_results['Tiempo (s)']):
        plt.text(i, v + 0.1, f"{v:.2f} s", ha='center', fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.show()


In [6]:
if 'models' in locals():
    for name, m in models.items():
        y_p = m.predict(X_val_fusion_scaled)
        cm = confusion_matrix(y_val, y_p)
        plt.figure(figsize=(6, 5))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Podrida (0)', 'Fresca (1)'])
        disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
        plt.title(f"Matriz de Confusión B2B - {name}", fontweight='bold')
        plt.xlabel("Predicción del Escáner")
        plt.ylabel("Estado Real de la Manzana")
        plt.tight_layout()
        plt.show()


## 5. Mapas Visuales de Características (El Escáner por Dentro)


In [7]:
if 'X_val' in locals():
    idx_fresca = np.where(y_val == 1)[0][0]
    idx_podrida = np.where(y_val == 0)[0][0]
    
    img_fresca_gray = cv2.cvtColor((X_val[idx_fresca] * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
    img_podrida_gray = cv2.cvtColor((X_val[idx_podrida] * 255).astype("uint8"), cv2.COLOR_RGB2GRAY)
    
    _, hog_fresca_vis = hog(img_fresca_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    _, hog_podrida_vis = hog(img_podrida_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    
    lbp_fresca_vis = local_binary_pattern(img_fresca_gray, P=8, R=1, method="uniform")
    lbp_podrida_vis = local_binary_pattern(img_podrida_gray, P=8, R=1, method="uniform")
    
    fresca_q = (img_fresca_gray // 4).astype(np.uint8)
    podrida_q = (img_podrida_gray // 4).astype(np.uint8)
    glcm_fresca_vis = graycomatrix(fresca_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)[:, :, 0, 0]
    glcm_podrida_vis = graycomatrix(podrida_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)[:, :, 0, 0]
    
    fig, axes = plt.subplots(4, 2, figsize=(10, 12))
    
    axes[0, 0].imshow(X_val[idx_fresca])
    axes[0, 0].set_title("Manzana Óptima (Original)", fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(X_val[idx_podrida])
    axes[0, 1].set_title("Manzana Defectuosa (Original)", fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[1, 0].imshow(hog_fresca_vis, cmap='gray')
    axes[1, 0].set_title("HOG - Mapa de Gradientes")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(hog_podrida_vis, cmap='gray')
    axes[1, 1].set_title("HOG - Mapa de Gradientes")
    axes[1, 1].axis('off')
    
    axes[2, 0].imshow(lbp_fresca_vis, cmap='gray')
    axes[2, 0].set_title("LBP - Micro-textura")
    axes[2, 0].axis('off')
    
    axes[2, 1].imshow(lbp_podrida_vis, cmap='gray')
    axes[2, 1].set_title("LBP - Micro-textura")
    axes[2, 1].axis('off')
    
    axes[3, 0].imshow(glcm_fresca_vis, cmap='viridis')
    axes[3, 0].set_title("GLCM - Matriz Coocurrencia 0°")
    axes[3, 1].imshow(glcm_podrida_vis, cmap='viridis')
    axes[3, 1].set_title("GLCM - Matriz Coocurrencia 0°")
    
    plt.tight_layout()
    plt.show()


## 6. Simulación de Escáner Industrial en Tiempo Real


In [8]:
def escanear_manzana_industrial(img_rgb, modelo, scaler):
    t_start = time.time()
    feat_hog = extraer_hog([img_rgb])
    feat_lbp = extraer_lbp([img_rgb])
    feat_glcm = extraer_glcm([img_rgb])
    feat_fusion = np.hstack((feat_hog, feat_lbp, feat_glcm))
    feat_scaled = scaler.transform(feat_fusion)
    prediccion = modelo.predict(feat_scaled)[0]
    probabilidades = modelo.predict_proba(feat_scaled)[0]
    t_latency_ms = (time.time() - t_start) * 1000
    
    print("\n" + "=" * 65)
    print("ESCANER B2B - RESULTADO DE INSPECCIÓN EN TIEMPO REAL")
    print(f"Tiempo de respuesta de la celda de visión: {t_latency_ms:.2f} ms")
    
    if prediccion == 1:
        print(f"🟢 [ESTADO: ÓPTIMO / FRESCA] - Confianza: {probabilidades[1]*100:.1f}%")
        print("  -> Orden PLC: APROBADA para empaque marítimo.")
    else:
        print(f"🔴 [ESTADO: DEFECTUOSO / PODRIDA] - Confianza: {probabilidades[0]*100:.1f}%")
        print("  -> Orden PLC: RECHAZADA. Activando electroválvula neumática.")
    print("=" * 65)

if 'rf_model' not in locals() and 'models' in locals():
    rf_model = models['Random Forest']

if 'X_val' in locals():
    print("\n--- PROBANDO ESCÁNER INDUSTRIAL EN TIEMPO REAL ---")
    print("\n[Prueba Muestra A: Inspeccionando Manzana de la Banda]")
    escanear_manzana_industrial(X_val[idx_fresca], rf_model, scaler)
    print("\n[Prueba Muestra B: Inspeccionando Manzana de la Banda]")
    escanear_manzana_industrial(X_val[idx_podrida], rf_model, scaler)
